# 8. Further Examples

Simulation examples beyond validation.

- Squeeze-Out
- Customized Constitutive Laws (equation of state, wall stress)

## Squeeze-Out

A parabolic plate approaches a soft elastic surface ($E = 1$ MPa) at constant velocity $V$. The gap height of the undeformed geometry is

$$h(x,t) = H_0 - V t + A \left(x - x_c\right)^2, \qquad x_c = L_x/2,$$

with $H_0 = 30\,\mu \text{m}$, $V = 5$ mm/s and curvature $A = 0.25\,\mathrm{m^{-1}}$. Ambient pressure is prescribed at both ends.

The time-dependent geometry cannot be specified in the YAML file. It is set from the Python script below by a callback that updates the gap height after every time step.

$\rightarrow$ **[YAML File](../../config_lib/validation/parabolic_squeeze_elastic.yaml)**

In [ ]:
from GaPFlow.problem import Problem

H0, V, A = 30.0e-6, 5.0e-3, 25.0e-2

problem = Problem.from_yaml('../../config_lib/validation/parabolic_squeeze_elastic.yaml')
x_center = problem.grid['Lx'] / 2


def set_gap():
    problem.topo.set_mapped_height(H0 - V * problem.simtime + A * (problem.topo.xx - x_center) ** 2)

set_gap()  # this is annoying, will be fixed
problem.add_callback(set_gap)
problem.run()

In [ ]:
problem.animate()

## Customized Equation of State

A user-defined equation of state is registered with <code style="color:#c586c0">problem.set_custom_eos(function)</code>. The function takes the density $\rho$ (and optionally the gap height $h$) as scalars and returns the pressure in SI units.

Inclined slider with a compressible gas. Three cases are compared:

- built-in law
- custom law (same as built-in for validation)
- custom law with gap-height dependency - illustrative example without physical background

$\rightarrow$ **[YAML File](../../config_lib/validation/inclined_slider.yaml)**

In [ ]:
from GaPFlow.problem import Problem
from utils.plotting import plot_custom_eos_comparison
from utils.solver_comparison import get_profile_1d

P0, rho0, h_min = 101325., 1.1853, 1.0e-5


def ideal_gas(rho):
    return P0 * rho / rho0


def gap_dependent(rho, h):
    return P0 * (rho / rho0) ** (1.0 + h_min / h)


results = {}
for label, eos in [('built-in', None), ('custom, same law', ideal_gas), ('custom, gap-dependent', gap_dependent)]:
    problem = Problem.from_yaml('../../config_lib/validation/inclined_slider.yaml')
    if eos is not None:
        problem.set_custom_eos(eos)  # the magic happens here
    problem.run()
    results[label] = get_profile_1d(problem)

plot_custom_eos_comparison(results)

# Customized Wall Stress

A user-defined wall stress is registered with <code style="color:#c586c0">problem.set_custom_wall_stress(function)</code>. The function takes the 
- density $\rho$ [$\text{kg}/\text{m}^3$],
- cavity fraction $\theta$ ($\rho_{\text{eff}} = \rho (1 - \theta)$) [-],
- mass flux component $j_i$ in [$\text{kg}/\text{m}^2 \, \text{s}$],
- gap height $h$ in [$\text{m}$], and 
- wall velocities $U_{i,\mathrm{bot}}$, $U_{i,\mathrm{top}}$ [$\text{m}/\text{s}$]

with $i \in \{x,y\}$ in the direction of the stress. The function needs to return two scalars: the shear stresses on the bottom and top wall in [$\text{Pa}$].

So the user provides only one mapping $\rho, \theta, j_i, h, U_{i, \text{bot}}, U_{i, \text{top}} \mapsto \tau_{iz,\text{bot}}, \tau_{iz,\text{top}}$ and internally, this is applied to both $x,y$ directions.


Pressure or viscosity are not passed, so the function needs to build them from $\rho$ itself. This allows a modular setup, where each constitutive law is a separate function and the laws are chained:

$$\rho \;\xrightarrow{\text{EoS}}\; p \;\xrightarrow{\text{Barus}}\; \eta \;\xrightarrow{\;\rho,\,j,\,h,\,U\;}\; \tau_\mathrm{bot},\, \tau_\mathrm{top}$$

For demonstration, we simulate a parabolic slider using DH EoS and Barus piezoviscosity. The wall stress uses the no-slip Couette-Poiseuille solution for a gap-averaged velocity $\bar{u} = j / \rho$,

$$\tau_\mathrm{bot} = \frac{2\eta}{h}\left(3\bar{u} - 2U_\mathrm{bot} - U_\mathrm{top}\right), \qquad \tau_\mathrm{top} = \frac{2\eta}{h}\left(U_\mathrm{bot} + 2U_\mathrm{top} - 3\bar{u}\right).$$

$\rightarrow$ **[YAML File](../../config_lib/validation/journal_bearing.yaml)**

In [ ]:
from GaPFlow.problem import Problem
import jax.numpy as jnp
from utils.plotting import plot_custom_wall_stress

P0, rho0, C1, C2 = 101325., 877.7007, 3.5e10, 1.23
eta0, alpha = 0.039, 2.0e-8


def eos(rho):
    return P0 + C1 * (rho / rho0 - 1.0) / (C2 - rho / rho0)


def barus(p):
    return eta0 * jnp.exp(alpha * p)


def wall_stress(rho, theta, j, h, U_bot, U_top):
    u_mean = j / (rho * (1.0 - theta))
    eta = barus(eos(rho)) * (1.0 - theta)
    tau_bot = 2.0 * eta / h * (3.0 * u_mean - 2.0 * U_bot - U_top)
    tau_top = 2.0 * eta / h * (U_bot + 2.0 * U_top - 3.0 * u_mean)
    return tau_bot, tau_top


problem = Problem.from_yaml('../../config_lib/validation/journal_bearing.yaml')
problem.set_custom_eos(eos)
problem.set_custom_wall_stress(wall_stress)
problem.run()

plot_custom_wall_stress(problem)